# പാഠം 11 - മോഡൽ കോൺ്റെക്സ്റ്റ് പ്രോട്ടോക്കോൾ (MCP)

**മോഡൽ കോൺ്റെക്സ്റ്റ് പ്രോട്ടോക്കോൾ (MCP)** ലഘുലേഖകൾക്ക് ടൂളുകൾ, മൂല്യങ്ങൾ, ഡാറ്റാ സ്രോതസ്സ് എന്നിവ-runtime-ൽ ഡൈനാമിക്കായി കണ്ടെത്തുകയും ഉപയോഗിക്കുകയും ചെയ്യാൻ അനുവദിക്കുന്ന ഒരു തുറന്ന സ്റ്റാൻഡേർഡാണ്. ഒരു ഏജന്റിൽ ടൂളുകൾ ഹാർഡ്‌കോഡ് ചെയ്യുന്നതിനുപകരം, MCP ഏജന്റുകൾ അവശ്യമായ ശേഷികൾ പ്രദർശിപ്പിക്കുന്ന ബാഹ്യ സർവീസുകളുമായി ചേരാനുള്ള വഴി നൽകുന്നു.

ഈ പാഠത്തിൽ, നിങ്ങൾ അറിയുക:
- MCP-എന്താണ്, ഏജന്റ് സിസ്റ്റങ്ങൾക്കുള്ള പ്രാധാന്യം എന്ത്
- MCP ക്ലയന്റ്-സർവർ വാസ്തവിക നിർമ്മിതിയിൽ എങ്ങനെ പ്രവർത്തിക്കുന്നു
- MCP ശൈലിയിലെ ടൂൾ കണ്ടെത്തൽ ഉപയോഗിക്കുന്ന ഏജന്റുകൾ എങ്ങനെ നിർമിക്കാം


## ക്രമീകരണം

**ആവശ്യമായ മുൻ‌വശങ്ങൾ:**
- മോഡൽ വിന്യസിച്ചിട്ടുള്ള Microsoft Foundry പ്രോജക്റ്റ്
- അംഗീകാരത്തിനായി `az login` നടത്തുക


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## മോഡൽ കോൺടെക്സ് പ്രോട്ടോക്കോൾ (MCP) എന്താണ്?

MCP എഐ ഏجن്റുകൾക്ക് ബാഹ്യ ഉപകരണങ്ങളും ഡേറ്റാ സ്രോതസുകളും കണ്ടെത്താനും ഇടപഴകാനും ഒരു സ്റ്റാൻഡേർഡ് മാർഗ്ഗം നിർവചിക്കുന്നു:

- **MCP സർവർ**: സ്റ്റാൻഡേർഡ് പ്രോട്ടോക്കോൾ വഴി ഉപകരണങ്ങൾ, വിഭവങ്ങൾ, പ്രോപ്റ്റുകൾ ഒരുങ്ങുന്നു
- **MCP ക്ലയന്റ്**: സർവറുകളിൽ ചേരുകയും ലഭ്യമായ ശേഷികൾ കണ്ടെത്തുകയും ചെയ്യുന്ന ഏജന്റിന്റെ റൺടൈം
- **ഡൈനാമിക് ഡിസ്ക്കവറി**: ഏജന്റുകൾക്ക് കേഡ്ഡ് ചെയ്ത ഉപകരണങ്ങൾ വേണ്ട — അവയ്ക്ക് റൺടൈമിൽ ലഭ്യമായതെന്താണെന്ന് കണ്ടെത്താം

ഇത് ഏജന്റ് കോഡ് മാറ്റാതെ പുതിയ ശേഷികൾ ചേർക്കാവുന്ന വിപുലീകരണയോഗ്യമായ ഏജന്റ് സിസ്റ്റങ്ങൾ നിർമ്മിക്കാൻ ശക്തിയാണ്.


## MCP എങ്ങനെ പ്രവർത്തിക്കുന്നു

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ഏജന്റ് (MCP ക്ലയന്റ്) ഒരു MCP സെർവർക്ക് കണക്ട് ചെയ്യും
2. സെർവർ ലഭ്യമായ ടൂളുകളും അവയുടെ സ്കീമകളും അടങ്ങിയ ഒരു ലിസ്റ്റ് അയക്കുന്നു
3. തുടർന്ന് ഏജന്റ് അതിന്റെ നാമനിർണ്ണയത്തിനിടെ കണ്ടെത്തിയ ഏതെങ്കിലും ടൂൾ വിളിച്ചുപയോഗിക്കാം
4. ഫലം അതേ പ്രോട്ടോക്കോൾ വഴി വന്നു ചെലുത്തപ്പെടുന്നു


## MCP ടൂൾ കണ്ടെത്തലിന്റെ സിമുലേഷൻ

ഒരു യഥാർത്ഥ MCP സെർവർ പ്രവർത്തനക്ഷമമായ സെർവർ പ്രോസസ്സ് ആവശ്യമായതിനാൽ, MCP-യുമായുള്ള ബന്ധമുണ്ടായ ആകോമൊഡേഷൻ സേവനം നൽകുന്നതിനെ സിമുലേറ്റ് ചെയ്യുന്ന `@tool` ഫംഗ്ഷനുകൾ ഉപയോഗിച്ച് നമുക്ക് ഈ മാതൃക കാണിക്കാം.

പ്രൊഡക്ഷനിൽ, ഈ ടൂളുകൾ പ്രാദേശികമായി നിർവചിക്കപ്പെടുന്നതിനു പകരം MCP സെർവറിൽ നിന്ന് ഡൈനാമിക്കായി കണ്ടെത്തപ്പെടും.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-ശൈലിയുടെ ടൂൾസുകളുമായി ഒരു ഏജന്റ് നിർമ്മിക്കൽ


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ഉത്പാദനത്തിൽ MCP

ഉത്പാദന പരിസരത്ത്, MCP ശക്തമായ മാതൃകകൾ സജ്ജമാക്കുന്നു:

- **ഡൈനാമിക് ടൂൾ കണ്ടെത്തൽ**: ഏജന്റുകൾ MCP സെർവർകളോട് ബന്ധപ്പെടുകയും റൺടൈമിൽ ടൂളുകൾ കണ്ടെത്തുകയും ചെയ്യുന്നു
- **അാൽപ്പെട്ട ആർക്കിടെക്ചർ**: ടൂൾ പ്രൊവൈഡറുകൾ ഏജന്റിനേക്കാൾ സ്വതന്ത്രമായ അപ്‌ഡേറ്റ് നടത്താൻ കഴിയും
- **സംഘടനകൾക്ക് മുകളിലൂടെ പങ്കിടൽ**: ടീമുകൾ MCP സെർവർ മാർഗത്തിലൂടെ കാര്യങ്ങൾ പങ്കുവെക്കാം, ഏത് ഏജന്റും ഉപയോഗിക്കാനാകുന്നവ
- **Microsoft Agent Framework പിന്തുണ**: MAF ല് നിർമിത MCP ക്ലയന്റ് പിന്തുണ `mcp` ഇന്റഗ്രേഷൻ മുഖേനയും ഉണ്ട്

യഥാർത്ഥ MCP സെർവർ MAF ഉപയോഗിച്ച് ഉപയോഗിക്കാൻ, `hosted_mcp_tool()` അല്ലെങ്കിൽ MCP ക്ലയന്റ് ഇന്റഗ്രേഷൻ വഴി ബന്ധപെടും.

**കൂടുതൽ അറിയുക:**
- [MCP വിശദീകരണം](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP പിന്തുണ](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## സംഗ്രഹം

ഈ പാഠത്തിൽ, നിങ്ങൾ തുടങ്ങിയுது:
- **MCP** ഏജന്റുകളും ടൂൾ പ്രൊവൈഡർമാരും തമ്മിലുള്ള ഡൈനാമിക് ടൂൾ കണ്ടെത്തലിനുള്ള ഒരു തുറന്ന സ്റ്റാൻഡേർഡാണ്
- **ക്ലയന്റ്-സർവർ ആർക്കിടെക്ചർ** ഏജന്റുകൾക്ക് റൺടൈമിൽ കഴിവുകൾ കണ്ടെത്താൻ അനുവദിക്കുന്നു
- MCP കോഡ് മാറ്റങ്ങൾ നടത്താതെ ടൂളുകൾ ചേർക്കാനാകുന്ന **വിപുലമാകുന്ന, വ്യത്യസ്തമായ ഏജന്റ് സിസ്റ്റങ്ങൾ** സജ്ജമാക്കുന്നു
- Microsoft Agent Framework ഉൽ‌പാദന ഉപയോഗത്തിനായി **അകമ്പടി MCP പിന്തുണ** നൽകുന്നു


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
